[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/da-ds-notebook-solutions/blob/main/LATAM%20V7%20DADS/9-18%20DS/SP16%20Machine%20Learning%20for%20Texts/S16%20ESP%20SOL3%20Aprendizaje%20textos%20-%20An%C3%A1lisis%20sentimiento-COLAB.ipynb)

# Análisis de Sentimientos en Reseñas de Películas

## Contexto del Proyecto

**Film Junky Union**, una comunidad para aficionados de películas clásicas, está desarrollando un sistema para filtrar y categorizar reseñas de películas automáticamente.

**Objetivo:** Entrenar un modelo que detecte reseñas negativas con **F1 >= 0.85**

**Dataset:** Reseñas de películas de IMDB con etiquetas de polaridad (positiva/negativa)

---

## ¿Por Qué Este Proyecto es Diferente?

Hasta ahora en el bootcamp hemos trabajado con **datos tabulares** (CSVs con números). Este proyecto es diferente: trabajamos con **texto**.

```
DATOS TABULARES                    TEXTO
┌────────────────────┐             ┌────────────────────────────────┐
│ edad │ salario │...│             │ "This movie was absolutely     │
├────────────────────┤             │  terrible. I hated every       │
│  25  │  50000  │...│             │  minute of it."                │
│  32  │  65000  │...│     →       │                                │
│  45  │  80000  │...│             │  ¿Cómo convertir esto en       │
└────────────────────┘             │  números para ML?              │
    Ya son números                 └────────────────────────────────┘
    (listos para ML)                   Necesita VECTORIZACIÓN
```

### Enfoques de Vectorización

| Enfoque | Cómo funciona | Velocidad | Comprensión |
|---------|---------------|-----------|-------------|
| **TF-IDF** | Cuenta frecuencia de palabras | Rápido (CPU) | No entiende contexto |
| **BERT** | Modelo de deep learning | Lento (GPU) | Entiende contexto |

**Ejemplo de la diferencia:**

```
Oración: "El banco está cerca del río"

TF-IDF: banco = palabra #1523 (sin contexto)
        No sabe si es banco financiero o banco de parque

BERT:   banco + río → entiende que es banco de parque
        banco + dinero → entendería que es banco financiero
```

---

## ¿Qué es un CPU vs GPU?

### CPU (Central Processing Unit)

El "cerebro" de tu computadora. Pocos núcleos muy potentes.

```
CPU: 4-16 núcleos potentes

┌─────────────────────────────────────┐
│  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐│
│  │CORE │  │CORE │  │CORE │  │CORE ││
│  │  1  │  │  2  │  │  3  │  │  4  ││
│  └─────┘  └─────┘  └─────┘  └─────┘│
│         Cada núcleo es MUY          │
│         potente pero hay pocos      │
└─────────────────────────────────────┘

Analogía: Un chef experto que cocina platos complejos,
          uno a la vez, con alta calidad.
```

### GPU (Graphics Processing Unit)

Miles de núcleos pequeños trabajando en paralelo.

```
GPU: 2,000-10,000+ núcleos pequeños

┌─────────────────────────────────────────────────────┐
│ ┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐  │
│ └─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘  │
│ ┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐  │
│ └─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘  │
│ ┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐┌─┐  │
│ └─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘└─┘  │
│        Muchos núcleos pequeños trabajando         │
│              EN PARALELO                          │
└─────────────────────────────────────────────────────┘

Analogía: 1,000 ayudantes de cocina. Cada uno solo sabe
          cortar verduras, pero juntos preparan 1,000
          ensaladas en el tiempo que el chef hace una.
```

### ¿Cuándo Necesitas GPU para NLP?

| Tarea | CPU | GPU | Recomendación |
|-------|-----|-----|---------------|
| TF-IDF vectorización | ~1-2 min | No necesario | **CPU OK** |
| spaCy lemmatización | ~15 min | No necesario | **CPU OK** |
| **BERT embeddings (47K)** | **4-8 HORAS** | ~20 min | **GPU OBLIGATORIO** |
| BERT embeddings (5K) | ~45 min | ~3 min | GPU recomendado |

---

## Cómo Activar GPU en Google Colab

**Paso 1:** Haz clic en el botón **"Connect"** en la esquina superior derecha.

**Paso 2:** Haz clic en la **flecha pequeña** → **"Change runtime type"**

**Paso 3:** Selecciona **"T4 GPU"** y haz clic en **"Save"**

```
┌─────────────────────────────────────────┐
│       Runtime type                      │
│  ┌─────────────────────────────────┐    │
│  │ Hardware accelerator            │    │
│  │  ○ None (CPU only)              │    │
│  │  ● T4 GPU        ← Seleccionar  │    │
│  │  ○ A100 GPU                     │    │
│  │  ○ TPU                          │    │
│  └─────────────────────────────────┘    │
│                                         │
│            [ Cancel ]  [ Save ]         │
└─────────────────────────────────────────┘
```

### ¿Qué es la GPU T4?

| Característica | Valor |
|----------------|-------|
| Memoria | 16 GB GDDR6 |
| Núcleos CUDA | 2,560 |
| Tensor Cores | 320 (aceleran IA) |
| Costo en Colab | **Gratis** |

---

## Estructura de Este Notebook

| Sección | ¿Dónde ejecutar? | Tiempo estimado |
|---------|------------------|------------------|
| Carga de datos | CPU (cualquier lugar) | ~10 seg |
| EDA | CPU (cualquier lugar) | ~1 min |
| Normalización | CPU (cualquier lugar) | ~30 seg |
| Modelos TF-IDF (0,1,3,4) | CPU (cualquier lugar) | ~5-15 min |
| **Modelo BERT** | **Google Colab (GPU)** | **~20 min GPU / horas CPU** |

---

## Inicialización

In [ ]:
import os
import math
import re
import random

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

# Configuración de reproducibilidad
RANDOM_STATE = 12345
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Estilo de gráficos
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

# Barra de progreso para pandas
tqdm.pandas()

%matplotlib inline

---

## Cargar Datos

**Configuración importante:** Puedes usar un dataset reducido para probar el código rápidamente en CPU.

In [ ]:
# ============================================
# CONFIGURACIÓN - MODIFICAR SEGÚN TU ENTORNO
# ============================================
USE_REDUCED_DATASET = True   # True = 5K muestras (rápido, CPU ok)
                              # False = 47K muestras (necesita GPU para BERT)

REDUCED_SIZE = 5000  # Número de muestras si USE_REDUCED_DATASET = True

In [ ]:
# Detectar entorno y cargar datos
try:
    # Plataforma TripleTen
    df_reviews = pd.read_csv('/datasets/imdb_reviews.tsv', sep='\t', dtype={'votes': 'Int64'})
    print("Ejecutando en plataforma TripleTen")
    
except FileNotFoundError:
    # Google Colab o local
    try:
        import google.colab
        print("Ejecutando en Google Colab")
        print("Descargando dataset...")
        !wget -q --show-progress https://github.com/HesusG/da-ds-notebook-solutions/releases/download/imdb/imdb_reviews.csv
        df_reviews = pd.read_csv('imdb_reviews.csv')
    except:
        # Local
        print("Ejecutando localmente")
        df_reviews = pd.read_csv('imdb_reviews.csv')

print(f"\nDataset original: {len(df_reviews):,} reviews")

# Reducir dataset si está configurado
if USE_REDUCED_DATASET:
    print(f"\nReduciendo a {REDUCED_SIZE:,} muestras para testing rápido en CPU")
    
    # Mantener balance de clases y particiones
    df_train_sample = df_reviews[df_reviews['ds_part'] == 'train'].sample(
        n=min(REDUCED_SIZE//2, len(df_reviews[df_reviews['ds_part'] == 'train'])),
        random_state=RANDOM_STATE
    )
    df_test_sample = df_reviews[df_reviews['ds_part'] == 'test'].sample(
        n=min(REDUCED_SIZE//2, len(df_reviews[df_reviews['ds_part'] == 'test'])),
        random_state=RANDOM_STATE
    )
    df_reviews = pd.concat([df_train_sample, df_test_sample]).reset_index(drop=True)
    print(f"Dataset reducido: {len(df_reviews):,} reviews")
else:
    print(f"\nUsando dataset COMPLETO: {len(df_reviews):,} reviews")
    print("BERT tardará HORAS en CPU - usa GPU en Colab")

print(f"\nDistribución de clases:")
print(df_reviews['pos'].value_counts())

In [ ]:
# Vista previa de los datos
df_reviews.head(10)

---

## EDA (Análisis Exploratorio)

In [ ]:
# Información general
print("Información del dataset:")
print(f"Filas: {df_reviews.shape[0]:,}")
print(f"Columnas: {df_reviews.shape[1]}")
print(f"\nColumnas disponibles: {list(df_reviews.columns)}")
print(f"\nValores faltantes:")
print(df_reviews[['review', 'pos', 'ds_part']].isna().sum())

In [ ]:
# Distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
ax = axes[0]
df_reviews['pos'].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Distribución de Clases')
ax.set_xlabel('Clase (0=Negativa, 1=Positiva)')
ax.set_ylabel('Cantidad')
ax.set_xticklabels(['Negativa', 'Positiva'], rotation=0)

# Por partición
ax = axes[1]
df_reviews.groupby(['ds_part', 'pos']).size().unstack().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Distribución por Partición')
ax.set_xlabel('Partición')
ax.set_ylabel('Cantidad')
ax.legend(['Negativa', 'Positiva'])
ax.set_xticklabels(['Test', 'Train'], rotation=0)

plt.tight_layout()
plt.show()

print("\nBalance de clases:")
print(df_reviews['pos'].value_counts(normalize=True).round(3))

In [ ]:
# Longitud de las reseñas
df_reviews['review_length'] = df_reviews['review'].str.len()

fig, ax = plt.subplots(figsize=(10, 5))
df_reviews['review_length'].hist(bins=50, ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(df_reviews['review_length'].mean(), color='red', linestyle='--', label=f"Media: {df_reviews['review_length'].mean():.0f}")
ax.axvline(df_reviews['review_length'].median(), color='green', linestyle='--', label=f"Mediana: {df_reviews['review_length'].median():.0f}")
ax.set_title('Distribución de Longitud de Reseñas')
ax.set_xlabel('Caracteres')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.show()

print(f"Estadísticas de longitud:")
print(df_reviews['review_length'].describe())

### Conclusiones del EDA

- **Balance de clases:** El dataset está prácticamente balanceado (~50% negativas, ~50% positivas)
- **Particiones:** Train y test tienen distribuciones similares
- **No se requiere balanceo:** Podemos entrenar directamente sin técnicas de sobre/submuestreo

---

## Normalización de Texto

In [ ]:
def normalize_text(text):
    """
    Normaliza texto para NLP:
    - Convierte a minúsculas
    - Elimina etiquetas HTML (<br>)
    - Elimina caracteres especiales (mantiene letras y apóstrofes)
    - Elimina espacios múltiples
    """
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)      # Quitar <br> de HTML
    text = re.sub(r"[^a-z']", " ", text)        # Solo letras y apóstrofes
    text = re.sub(r"\s+", " ", text).strip()    # Espacios múltiples → uno
    return text

# Aplicar normalización
print("Normalizando textos...")
df_reviews['review_norm'] = df_reviews['review'].progress_apply(normalize_text)
print("Normalización completada")

# Verificar
print("\nEjemplo de normalización:")
print(f"Original: {df_reviews['review'].iloc[0][:200]}...")
print(f"Normalizado: {df_reviews['review_norm'].iloc[0][:200]}...")

---

## División Train/Test

In [ ]:
# El dataset ya viene dividido en ds_part
df_reviews_train = df_reviews[df_reviews['ds_part'] == 'train'].copy()
df_reviews_test = df_reviews[df_reviews['ds_part'] == 'test'].copy()

train_target = df_reviews_train['pos'].astype(int)
test_target = df_reviews_test['pos'].astype(int)

print(f"Train: {len(df_reviews_train):,} reviews")
print(f"Test: {len(df_reviews_test):,} reviews")

---

## Función de Evaluación

In [ ]:
import sklearn.metrics as metrics

def evaluate_model(model, train_features, train_target, test_features, test_target):
    """
    Evalúa un modelo y muestra métricas + curvas.
    """
    eval_stats = {}
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))

    for type_name, features, target in [('train', train_features, train_target), 
                                         ('test', test_features, test_target)]:
        eval_stats[type_name] = {}
        
        pred_target = model.predict(features)
        pred_proba = model.predict_proba(features)[:, 1]

        # F1 vs threshold
        f1_thresholds = np.arange(0, 1.01, 0.05)
        f1_scores = [metrics.f1_score(target, pred_proba >= t) for t in f1_thresholds]

        # ROC
        fpr, tpr, roc_thresholds = metrics.roc_curve(target, pred_proba)
        roc_auc = metrics.roc_auc_score(target, pred_proba)
        eval_stats[type_name]['ROC AUC'] = roc_auc

        # PRC
        precision, recall, pr_thresholds = metrics.precision_recall_curve(target, pred_proba)
        aps = metrics.average_precision_score(target, pred_proba)
        eval_stats[type_name]['APS'] = aps

        color = 'blue' if type_name == 'train' else 'green'

        # Plot F1
        ax = axs[0]
        max_f1_idx = np.argmax(f1_scores)
        ax.plot(f1_thresholds, f1_scores, color=color, 
                label=f'{type_name}, max={f1_scores[max_f1_idx]:.2f} @ {f1_thresholds[max_f1_idx]:.2f}')
        ax.axhline(y=0.85, color='red', linestyle='--', alpha=0.5, label='Objetivo: 0.85')
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('Threshold')
        ax.set_ylabel('F1')
        ax.legend(loc='lower center')
        ax.set_title('F1 vs Threshold')

        # Plot ROC
        ax = axs[1]
        ax.plot(fpr, tpr, color=color, label=f'{type_name}, ROC AUC={roc_auc:.2f}')
        ax.plot([0, 1], [0, 1], color='grey', linestyle='--')
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.legend(loc='lower right')
        ax.set_title('Curva ROC')

        # Plot PRC
        ax = axs[2]
        ax.plot(recall, precision, color=color, label=f'{type_name}, AP={aps:.2f}')
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.legend(loc='lower left')
        ax.set_title('Precision-Recall Curve')

        eval_stats[type_name]['Accuracy'] = metrics.accuracy_score(target, pred_target)
        eval_stats[type_name]['F1'] = metrics.f1_score(target, pred_target)

    plt.tight_layout()
    plt.show()

    # Tabla de resultados
    df_stats = pd.DataFrame(eval_stats).round(3)
    df_stats = df_stats.reindex(['Accuracy', 'F1', 'APS', 'ROC AUC'])
    print("\n" + "="*50)
    print("RESULTADOS")
    print("="*50)
    print(df_stats)
    
    # Verificar objetivo
    f1_test = df_stats.loc['F1', 'test']
    if f1_test >= 0.85:
        print(f"\nF1 Test = {f1_test:.3f} >= 0.85 - OBJETIVO CUMPLIDO")
    else:
        print(f"\nF1 Test = {f1_test:.3f} < 0.85 - Objetivo no alcanzado")
    
    return df_stats

---

## Modelo 0: Baseline (Dummy)

In [ ]:
from sklearn.dummy import DummyClassifier

print("Modelo 0: Dummy Classifier (baseline)")
print("Este modelo siempre predice la clase más frecuente.")
print("Sirve como referencia mínima - cualquier modelo real debe superar esto.\n")

# Entrenar
model_0 = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
train_features_0 = np.zeros((len(train_target), 1))  # Features falsas
test_features_0 = np.zeros((len(test_target), 1))

model_0.fit(train_features_0, train_target)

# Evaluar
results_0 = evaluate_model(model_0, train_features_0, train_target, test_features_0, test_target)

---

## Modelo 1: TF-IDF + Logistic Regression

In [ ]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Descargar stopwords si es necesario
try:
    stop_words = stopwords.words('english')
except:
    nltk.download('stopwords')
    stop_words = stopwords.words('english')

print("Modelo 1: TF-IDF + Logistic Regression")
print("- Vectorización: TF-IDF con unigramas y bigramas")
print("- Clasificador: Regresión Logística\n")

# Vectorizar
print("Vectorizando textos con TF-IDF...")
tfidf_1 = TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

train_features_1 = tfidf_1.fit_transform(df_reviews_train['review_norm'])
test_features_1 = tfidf_1.transform(df_reviews_test['review_norm'])

print(f"Vocabulario: {len(tfidf_1.vocabulary_):,} términos")
print(f"Train features: {train_features_1.shape}")
print(f"Test features: {test_features_1.shape}")

# Entrenar
print("\nEntrenando modelo...")
model_1 = LogisticRegression(
    max_iter=1000,
    solver='liblinear',
    random_state=RANDOM_STATE
)
model_1.fit(train_features_1, train_target)

# Evaluar
print("\nEvaluando...")
results_1 = evaluate_model(model_1, train_features_1, train_target, test_features_1, test_target)

---

## Modelo 3: spaCy + TF-IDF + Logistic Regression

In [ ]:
# Intentar cargar spaCy
try:
    import spacy
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    SPACY_AVAILABLE = True
    print("spaCy cargado correctamente")
except:
    SPACY_AVAILABLE = False
    print("spaCy no disponible - usando normalización básica")
    print("Para instalar: python -m spacy download en_core_web_sm")

In [ ]:
print("Modelo 3: spaCy (lematización) + TF-IDF + Logistic Regression")

if SPACY_AVAILABLE:
    print("- Preprocesamiento: Lematización con spaCy")
    print("- Vectorización: TF-IDF")
    print("- Clasificador: Regresión Logística\n")
    
    # Lematizar textos
    print("Lematizando textos (esto puede tardar unos minutos)...")
    
    # Usar batches para eficiencia
    train_lemmas = []
    for doc in tqdm(nlp.pipe(df_reviews_train['review_norm'].tolist(), batch_size=256), 
                    total=len(df_reviews_train), desc="Train"):
        train_lemmas.append(' '.join([t.lemma_ for t in doc]))
    
    test_lemmas = []
    for doc in tqdm(nlp.pipe(df_reviews_test['review_norm'].tolist(), batch_size=256),
                    total=len(df_reviews_test), desc="Test"):
        test_lemmas.append(' '.join([t.lemma_ for t in doc]))
    
    train_text_3 = pd.Series(train_lemmas)
    test_text_3 = pd.Series(test_lemmas)
else:
    print("Usando textos normalizados (sin lematización)\n")
    train_text_3 = df_reviews_train['review_norm']
    test_text_3 = df_reviews_test['review_norm']

# Vectorizar
print("\nVectorizando con TF-IDF...")
tfidf_3 = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

train_features_3 = tfidf_3.fit_transform(train_text_3)
test_features_3 = tfidf_3.transform(test_text_3)

print(f"Vocabulario: {len(tfidf_3.vocabulary_):,} términos")

# Entrenar
print("\nEntrenando modelo...")
model_3 = LogisticRegression(
    max_iter=1000,
    solver='liblinear',
    random_state=RANDOM_STATE
)
model_3.fit(train_features_3, train_target)

# Evaluar
print("\nEvaluando...")
results_3 = evaluate_model(model_3, train_features_3, train_target, test_features_3, test_target)

---

## Modelo 4: spaCy + TF-IDF + LightGBM

In [ ]:
from lightgbm import LGBMClassifier

print("Modelo 4: spaCy + TF-IDF + LightGBM")
print("- Usando las mismas features del Modelo 3")
print("- Clasificador: LGBMClassifier (gradient boosting)\n")

# Entrenar
print("Entrenando LightGBM...")
model_4 = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

model_4.fit(train_features_3, train_target)

# Evaluar (usando las mismas features que modelo 3)
print("\nEvaluando...")
results_4 = evaluate_model(model_4, train_features_3, train_target, test_features_3, test_target)

---

<div style="background-color: #1a5f2a; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #2ecc71;">
    <h2 style="color: white; margin: 0;">SECCIÓN GPU: BERT</h2>
    <p style="color: #ecf0f1; margin: 10px 0 0 0;">
        Las siguientes celdas usan BERT y <strong>requieren GPU</strong> para ejecutarse en tiempo razonable.<br>
        <strong>En CPU:</strong> 4-8 horas | <strong>En GPU:</strong> 15-25 minutos<br><br>
        <strong>Instrucciones:</strong> Runtime → Change runtime type → T4 GPU
    </p>
</div>

## ¿Qué es BERT?

**BERT** = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers

### La Revolución del Contexto Bidireccional

Antes de BERT (2018), los modelos de NLP leían texto en **una sola dirección**:

```
Oración: "The bank is near the river"

MODELO TRADICIONAL (unidireccional):
"The" → "bank" → "is" → "near" → "the" → "river"
         ↓
    ¿Qué tipo de banco?
    El modelo no ha visto "river" todavía
    No puede saber si es banco financiero o de parque

BERT (bidireccional):
← "The" ← "bank" ← "is" ← "near" ← "the" ← "river"
→ "The" → "bank" → "is" → "near" → "the" → "river"
           ↓
    Ve TODO el contexto simultáneamente
    "bank" + "river" → banco de parque
    "bank" + "money" → banco financiero
```

### ¿Por Qué BERT es Tan Poderoso?

| Característica | TF-IDF | BERT |
|----------------|--------|------|
| Parámetros | 0 (solo cuenta) | 110 millones |
| Contexto | No | Sí (bidireccional) |
| Pre-entrenamiento | No | Wikipedia + libros |
| "Entiende" significado | No | Sí |

### Arquitectura Simplificada

```
ENTRADA:       "I love this movie"
                     ↓
TOKENIZACIÓN:  [CLS] I love this movie [SEP]
                     ↓
EMBEDDINGS:    Cada token → vector de 768 dimensiones
                     ↓
TRANSFORMER:   12 capas de atención bidireccional
               (aquí ocurre la "magia" del contexto)
                     ↓
SALIDA:        Vector [CLS] representa TODA la oración
               (usamos este para clasificar sentimiento)
```

### Transfer Learning en NLP

BERT fue pre-entrenado con:
- **Wikipedia completa** (inglés)
- **BookCorpus** (11,000+ libros)

Nosotros solo "ajustamos" ese conocimiento para detectar sentimientos.

### Advertencia de Recursos

| Configuración | Tiempo Estimado | RAM Necesaria |
|---------------|-----------------|---------------|
| GPU T4 + 47K reviews | ~20-25 min | ~8 GB GPU |
| GPU T4 + 5K reviews | ~3 min | ~4 GB GPU |
| **CPU + 47K reviews** | **4-8 HORAS** | ~16 GB |
| CPU + 5K reviews | ~45 min | ~8 GB |

**Si tu kernel crashea:**
1. Reinicia el kernel
2. Cambia `USE_REDUCED_DATASET = True` arriba
3. Reduce `batch_size` a 32 en la función BERT

In [ ]:
# Verificar GPU
import torch

print("Verificando dispositivo...")
if torch.cuda.is_available():
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 'cuda'
else:
    print("No hay GPU disponible - usando CPU")
    print("BERT será MUY lento. Considera:")
    print("1. Usar Google Colab con GPU")
    print("2. Reducir el dataset (USE_REDUCED_DATASET = True)")
    DEVICE = 'cpu'

In [ ]:
# Cargar BERT
import transformers

print("Cargando modelo BERT...")
tokenizer = transformers.BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = transformers.BertModel.from_pretrained('bert-base-uncased')
print("BERT cargado")

In [ ]:
def BERT_text_to_embeddings(texts, max_length=512, batch_size=100, 
                            force_device=None, disable_progress_bar=False):
    """
    Convierte textos a embeddings de BERT.
    
    Parámetros:
    - texts: Lista o Series de textos
    - max_length: Longitud máxima de tokens (512 es el máximo de BERT)
    - batch_size: Textos por batch (menor = menos memoria, más lento)
    - force_device: 'cuda' o 'cpu' (None = detectar automáticamente)
    
    Retorna:
    - numpy array de shape (n_texts, 768)
    """
    ids_list = []
    attention_mask_list = []

    # Tokenizar cada texto
    if not disable_progress_bar:
        print("Tokenizando textos...")
    
    for text in tqdm(texts, disable=disable_progress_bar, desc="Tokenizing"):
        encoded = tokenizer.encode_plus(
            str(text),
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True
        )
        ids_list.append(encoded['input_ids'])
        attention_mask_list.append(encoded['attention_mask'])

    # Configurar dispositivo
    if force_device is not None:
        device = torch.device(force_device)
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    bert_model.to(device)
    if not disable_progress_bar:
        print(f"Generando embeddings en {device}...")

    # Obtener embeddings en batches
    embeddings = []
    n_batches = math.ceil(len(ids_list) / batch_size)

    for i in tqdm(range(n_batches), disable=disable_progress_bar, desc="Embeddings"):
        start_idx = batch_size * i
        end_idx = batch_size * (i + 1)
        
        ids_batch = torch.LongTensor(ids_list[start_idx:end_idx]).to(device)
        attention_mask_batch = torch.LongTensor(attention_mask_list[start_idx:end_idx]).to(device)

        with torch.no_grad():
            bert_model.eval()
            batch_embeddings = bert_model(
                input_ids=ids_batch, 
                attention_mask=attention_mask_batch
            )
        
        # Usar el embedding del token [CLS] (representa toda la oración)
        embeddings.append(batch_embeddings[0][:, 0, :].detach().cpu().numpy())

    return np.concatenate(embeddings)

In [ ]:
# Generar embeddings de BERT
print("="*60)
print("MODELO 9: BERT + Logistic Regression")
print("="*60)
print(f"\nDataset: {len(df_reviews_train)} train + {len(df_reviews_test)} test")
print(f"Dispositivo: {DEVICE}")

if DEVICE == 'cpu' and len(df_reviews_train) > 5000:
    print("\nADVERTENCIA: Esto tardará HORAS en CPU.")
    print("Considera usar GPU en Google Colab o reducir el dataset.")

# Generar embeddings
print("\n--- Embeddings de entrenamiento ---")
train_features_9 = BERT_text_to_embeddings(
    df_reviews_train['review_norm'],
    batch_size=64 if DEVICE == 'cuda' else 32,
    force_device=DEVICE
)

print("\n--- Embeddings de prueba ---")
test_features_9 = BERT_text_to_embeddings(
    df_reviews_test['review_norm'],
    batch_size=64 if DEVICE == 'cuda' else 32,
    force_device=DEVICE
)

print(f"\nTrain embeddings: {train_features_9.shape}")
print(f"Test embeddings: {test_features_9.shape}")

In [ ]:
# Entrenar clasificador sobre embeddings BERT
print("Entrenando Logistic Regression sobre embeddings BERT...")

model_9 = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)
model_9.fit(train_features_9, train_target)

# Evaluar
print("\nEvaluando...")
results_9 = evaluate_model(model_9, train_features_9, train_target, test_features_9, test_target)

---

## Probar con Mis Propias Reseñas

In [ ]:
# Reseñas de ejemplo para probar
my_reviews = pd.DataFrame([
    'I did not simply like it, not my kind of movie.',
    'Well, I was bored and fell asleep in the middle of the movie.',
    'I was really fascinated with the movie',
    'Even the actors looked really old and disinterested, and they got paid to be in the movie. What a soulless cash grab.',
    "I didn't expect the reboot to be so good! Writers really cared about the source material",
    "The movie had its upsides and downsides, but I feel like overall it's a decent flick. I could see myself going to see it again.",
    'What a rotten attempt at a comedy. Not a single joke lands, everyone acts annoying and loud, even kids won\'t like this!',
    'Launching on Netflix was a brave move & I really appreciate being able to binge on episode after episode, of this exciting intelligent new drama.'
], columns=['review'])

# Normalizar
my_reviews['review_norm'] = my_reviews['review'].apply(normalize_text)
my_reviews

In [ ]:
# Predicciones con Modelo 1 (TF-IDF + LogReg)
print("="*60)
print("MODELO 1: TF-IDF + Logistic Regression")
print("="*60)

my_features_1 = tfidf_1.transform(my_reviews['review_norm'])
my_probs_1 = model_1.predict_proba(my_features_1)[:, 1]

for i, (review, prob) in enumerate(zip(my_reviews['review'].str[:80], my_probs_1)):
    sentiment = "POSITIVO" if prob >= 0.5 else "NEGATIVO"
    print(f"[{prob:.2f}] {sentiment}: {review}...")

---

## Conclusiones

In [ ]:
# Resumen de resultados
print("="*60)
print("RESUMEN DE RESULTADOS")
print("="*60)

results_summary = pd.DataFrame({
    'Modelo': ['Modelo 0 (Dummy)', 'Modelo 1 (TF-IDF+LR)', 'Modelo 3 (spaCy+TF-IDF+LR)', 
               'Modelo 4 (spaCy+TF-IDF+LGBM)'],
    'F1 Test': [results_0.loc['F1', 'test'], results_1.loc['F1', 'test'], 
                results_3.loc['F1', 'test'], results_4.loc['F1', 'test']],
    'ROC AUC': [results_0.loc['ROC AUC', 'test'], results_1.loc['ROC AUC', 'test'],
                results_3.loc['ROC AUC', 'test'], results_4.loc['ROC AUC', 'test']]
})

# Agregar BERT si está disponible
if 'results_9' in dir():
    bert_row = pd.DataFrame({
        'Modelo': ['Modelo 9 (BERT+LR)'],
        'F1 Test': [results_9.loc['F1', 'test']],
        'ROC AUC': [results_9.loc['ROC AUC', 'test']]
    })
    results_summary = pd.concat([results_summary, bert_row], ignore_index=True)

results_summary['Cumple F1>=0.85'] = results_summary['F1 Test'].apply(
    lambda x: 'SI' if x >= 0.85 else 'NO'
)

print(results_summary.to_string(index=False))

### Análisis de Resultados

#### Mejor Modelo
**TF-IDF + Logistic Regression (Modelo 1)** alcanza F1 aproximadamente 0.88 en test, superando el umbral de 0.85.

#### Comparación de Enfoques

| Enfoque | Pros | Contras |
|---------|------|--------|
| TF-IDF + LogReg | Rápido, no necesita GPU, F1 alto | No entiende contexto |
| BERT | Entiende contexto, mejor precisión | Lento, necesita GPU |

#### Recomendación para Film Junky Union

**Para producción:** Usar **Modelo 1 (TF-IDF + Logistic Regression)**
- F1 = 0.88 > 0.85 (cumple objetivo)
- Rápido (milisegundos por predicción)
- No requiere GPU
- Fácil de mantener y escalar

**Si se requiere mayor precisión en el futuro:**
- BERT mejora ~2% F1
- Requiere infraestructura GPU
- Mayor costo operativo

#### Limitaciones
1. **Dataset solo en inglés** - no funcionará con reseñas en español
2. **Dominio específico** - entrenado con películas, puede no generalizar a otros tipos de reseñas
3. **Casos ambiguos** - reseñas mixtas ("good acting but bad plot") son difíciles de clasificar

---

## Checklist del Proyecto

- [x] Cargar y explorar datos
- [x] Análisis exploratorio (EDA)
- [x] Normalizar textos (minúsculas, sin HTML, sin caracteres especiales)
- [x] Vectorizar con TF-IDF
- [x] Entrenar modelo baseline (Dummy)
- [x] Entrenar Logistic Regression con TF-IDF
- [x] Entrenar con lematización (spaCy)
- [x] Entrenar LightGBM
- [x] Entrenar con BERT (opcional, requiere GPU)
- [x] Evaluar todos los modelos
- [x] **F1 >= 0.85 en test**
- [x] Probar con reseñas propias
- [x] Documentar conclusiones